In [1]:
!python --version

Python 3.11.14


In [2]:
import os, sys
import pandas as pd
import requests
import time

from pathlib import Path

ROOT = Path().resolve().parent.parent
SRC = os.path.join(ROOT, "src")

if str(SRC) not in sys.path:
    sys.path.append(str(SRC))

print("ROOT:", ROOT)
print("SRC added:", SRC)

from libs.calc_degs_lib import CALC_DEGS
from libs.tcga_gdc_lib import *
from libs.Basic import *


ROOT: /home/flavio/uv
SRC added: /home/flavio/uv/src


### Defaults

In [3]:
ROOT_DATA = '../data'
ROOT_DATA = Path(ROOT_DATA)

gdc = GDC(ROOT_DATA0=ROOT_DATA)

os.listdir(ROOT_DATA)[:10]


['cancer', 'reactome', 'vector_store', 'TCGA', 'GTEx', 'gdc_programs.txt']

In [4]:
os.listdir(gdc.root_data)[:10]

['app_main_local.py',
 'gtex_normal_tissue.ipynb',
 'docker_config.ipynb',
 'tcga_gdc_run_R_expression_dvlp.ipynb']

### Get all programs

In [5]:
force=False
verbose=True

prog_list = gdc.get_gdc_progams(force=force, verbose=verbose)

np.array(prog_list)


File read at '../data/gdc_programs.txt'


array(['TCGA', 'MATCH', 'TARGET', 'CGCI', 'CMI', 'APOLLO', 'BEATAML1.0',
       'CPTAC', 'MP2PRT', 'ALCHEMIST', 'CCDI', 'CCG', 'CDDP_EAGLE',
       'CTSP', 'EXCEPTIONAL_RESPONDERS', 'FM', 'HCMI', 'MMRF', 'NCICCR',
       'OHSU', 'ORGANOID', 'RC', 'REBC', 'TRIO', 'VAREPOP', 'WCDT'],
      dtype='<U22')

### Primary sites given a program

In [6]:
gdc.url_gdc_project

'https://api.self.cancer.gov/projects'

In [7]:
prog_id = 'TCGA'

df_psi = gdc.get_primary_sites(prog_id=prog_id, force=False, verbose=verbose)
print(len(df_psi))

df_psi.head(3)

Table opened ((33, 5)) at '../data/TCGA/primary_site_program_TCGA.tsv'
33


,psi_id,primary_site,project_id,disease_type,name
0,TCGA-ACC,Adrenal gland,TCGA-ACC,Adenomas and Adenocarcinomas,Adrenocortical Carcinoma
1,TCGA-PCPG,"Adrenal gland, Retroperitoneum and peritoneum,...",TCGA-PCPG,Paragangliomas and Glomus Tumors,Pheochromocytoma and Paraganglioma
2,TCGA-BLCA,Bladder,TCGA-BLCA,"Epithelial Neoplasms, NOS, Squamous Cell Neopl...",Bladder Urothelial Carcinoma


### GTEx

In [8]:
ROOT_GTEx = ROOT_DATA / Path('GTEx')

os.listdir(ROOT_GTEx)[:10]


['tcga_primary_site_to_gtex_ids.tsv']

In [9]:
fname_gtex = 'tcga_primary_site_to_gtex_ids.tsv'

dfg = pdreadcsv(fname_gtex, ROOT_GTEx)
print(len(dfg))

dfg.head(6)

33


,tcga_project_id,tcga_primary_site,gtex_dataset_id,preferred_gtex_id,gtex_tissue_site_detail_ids,mapping_confidence,notes,source_gtex_api
0,TCGA-ACC,Adrenal Gland,gtex_v8,Adrenal_Gland,Adrenal_Gland,high,Adrenocortical carcinoma; direct adrenal gland...,https://gtexportal.org/api/v2/redoc
1,TCGA-BLCA,Bladder,gtex_v8,Bladder,Bladder,high,"Direct bladder GTEx tissue, but GTEx bladder s...",https://gtexportal.org/api/v2/redoc
2,TCGA-BRCA,Breast,gtex_v8,Breast_Mammary_Tissue,Breast_Mammary_Tissue,high,Breast carcinoma; GTEx mammary/breast tissue.,https://gtexportal.org/api/v2/redoc
3,TCGA-CESC,Cervix Uteri,gtex_v8,Cervix_Ectocervix,Cervix_Ectocervix|Cervix_Endocervix|Uterus,medium,Cervical cancer may involve ectocervix/endocer...,https://gtexportal.org/api/v2/redoc
4,TCGA-CHOL,Bile Duct,gtex_v8,Liver,Liver,low,No direct bile duct/cholangiocyte GTEx tissue;...,https://gtexportal.org/api/v2/redoc
5,TCGA-COAD,Colon,gtex_v8,Colon_Transverse,Colon_Transverse|Colon_Sigmoid,high,Colon cancer; transverse/sigmoid are both vali...,https://gtexportal.org/api/v2/redoc


In [10]:
psi_id = 'TCGA-BRCA'
psi_id = 'TCGA-LUAD'

dfa = dfg[dfg.tcga_project_id == psi_id]

gtex_id = None

if len(dfa) == 1:
    row = dfa.iloc[0]

    gtex_id = row.preferred_gtex_id
    gtex_tissue_ids = row.gtex_tissue_site_detail_ids
    print(f"Found '{gtex_id}' tissue '{gtex_tissue_ids}'")
elif len(dfa) == 0:
    print("Not found")
else:
    print("Multiple matches found")
    for i, row in dfa.iterrows():
        gtex_id = row.preferred_gtex_id
        gtex_tissue_ids = row.gtex_tissue_site_detail_ids

        print(f"{row.tcga_project_id} -> '{gtex_id}' tissue '{gtex_tissue_ids}'")


Found 'Lung' tissue 'Lung'


### Get df_tumor

In [11]:
psi_id

'TCGA-LUAD'

In [12]:
verbose=False

dic_tumor, dic_normal = gdc.get_file_expression_tumor_and_normal(psi_id=psi_id, force=False, verbose=verbose)
df_tumor, df_normal = gdc.merge_normal_tumor_tables(dic_tumor, dic_normal,  imax_tumor=12, imax_normal=12, verbose=verbose)

len(df_tumor), len(df_normal)

>>> TCGA-LUAD
>>> TCGA-LUAD
0.........10.........20.........30.........40.........50.........60.........70.........80.........90......... -> 100
Dowloading tumor files: 0.........10.........20.........30.........40.........50.........60.........70.........80.........90.........100........ -> 109
>>> Processing normal data: 100
1 2 3 4 5 6 7 8 9 10 11 12 13 
>>> Processing tumor data: 11
1 2 3 4 5 6 7 8 9 10 11 

(150684, 240796)

In [13]:
df_tumor.head(3)

,gene_id,symbol,gene_type,tumor_1,tumor_2,tumor_3,tumor_4,tumor_5,tumor_6,tumor_7,tumor_8,tumor_9,tumor_10,tumor_11
0,ENSG00000000003,TSPAN6,protein_coding,1604,2392,1373,803,799,5095,2,792,1489,3870,522
1,ENSG00000000005,TNMD,protein_coding,0,0,8,2,1,0,14,0,0,1,0
2,ENSG00000000419,DPM1,protein_coding,459,816,713,799,400,3372,39,1043,345,1380,678


In [41]:
from scipy.stats import f_oneway

def add_entropy(df, read_limit:int=50, min_read:int=200, n_quantiles:int=10):
    # select tumor columns
    cols = [c for c in df.columns if c.startswith("tumor_")]

    def row_entropy(values):
        values = np.asarray(values, dtype=float)

        # remove NaNs
        values = values[np.isfinite(values)]

        if len(values) == 0:
            return np.nan

        # all equal → entropy = 0
        if np.allclose(values, values[0]):
            return 0.0

        # shift to positive (important if negative values exist)
        values = values - values.min()

        # avoid all zeros
        if np.allclose(values.sum(), 0):
            return 0.0

        probs = values / values.sum()

        # avoid log(0)
        probs = probs[probs > 0]

        entropy = -np.sum(probs * np.log2(probs))
        return entropy

    min_cols = int(len(cols) * 0.4)
    good = (df[cols] < read_limit).sum(axis=1) <= min_cols
    df = df[good].copy()

    df["total_sum"] = df[cols].sum(axis=1)

    df = df[df.total_sum > len(cols)*min_read]

    df["entropy"] = df[cols].apply(lambda row: row_entropy(row.values), axis=1)

    df["entropy_q"] = pd.qcut(df["entropy"], q=n_quantiles, labels=False, duplicates="drop" )

    return df


print(len(df_tumor))
df_tumor2 = add_entropy(df_tumor)
print(len(df_tumor2))
df_tumor2.head(3)


150684
15817


,gene_id,symbol,gene_type,tumor_1,tumor_2,tumor_3,tumor_4,tumor_5,tumor_6,tumor_7,tumor_8,tumor_9,tumor_10,tumor_11,entropy,entropy_q,total_sum
0,ENSG00000000003,TSPAN6,protein_coding,1604,2392,1373,803,799,5095,2,792,1489,3870,522,2.954896,5,18741
2,ENSG00000000419,DPM1,protein_coding,459,816,713,799,400,3372,39,1043,345,1380,678,2.911364,4,10044
3,ENSG00000000457,SCYL3,protein_coding,542,432,557,699,264,995,168,352,283,1706,471,2.852843,3,6469


In [46]:
def select_top_entropy(df, q=0.1):

    df = df[df.entropy_q <= q].copy()
    df = df.sort_values('entropy', ascending=True)
    df.reset_index(drop=True, inplace=True)

    return df

tumor_cols = [c for c in df_tumor.columns if c.startswith("tumor_")]

df_tumor3 = select_top_entropy(df_tumor2, q=0.1)
print(len(df_tumor3))
df_tumor3.head(3)

1582


,gene_id,symbol,gene_type,tumor_1,tumor_2,tumor_3,tumor_4,tumor_5,tumor_6,tumor_7,tumor_8,tumor_9,tumor_10,tumor_11,entropy,entropy_q,total_sum
0,ENSG00000135919,SERPINE2,protein_coding,189,821,1026,2040,1170,216535,5,599,193,1394,754,0.329341,0,224726
1,ENSG00000235162,C12orf75,protein_coding,627,56,231,253,47,22188,52,620,113,203,70,0.563498,0,24460
2,ENSG00000235174,RPL39P3,processed_pseudogene,73,87,72,89,12,13687,0,213,11,663,58,0.612907,0,14965


In [47]:
df_tumor3.head(30)

,gene_id,symbol,gene_type,tumor_1,tumor_2,tumor_3,tumor_4,tumor_5,tumor_6,tumor_7,tumor_8,tumor_9,tumor_10,tumor_11,entropy,entropy_q,total_sum
0,ENSG00000135919,SERPINE2,protein_coding,189,821,1026,2040,1170,216535,5,599,193,1394,754,0.329341,0,224726
1,ENSG00000235162,C12orf75,protein_coding,627,56,231,253,47,22188,52,620,113,203,70,0.563498,0,24460
2,ENSG00000235174,RPL39P3,processed_pseudogene,73,87,72,89,12,13687,0,213,11,663,58,0.612907,0,14965
3,ENSG00000080823,MOK,protein_coding,61,120,114,166,139,13949,15,93,79,424,112,0.614011,0,15272
4,ENSG00000101955,SRPX,protein_coding,52,600,238,82,183,16806,0,96,36,284,78,0.672237,0,18455
5,ENSG00000178031,ADAMTSL1,protein_coding,62,58,148,28,107,6498,10,10,53,170,82,0.673331,0,7226
6,ENSG00000164266,SPINK1,protein_coding,64067,83,10713,10,85,63,0,11,81,252,41,0.677665,0,75406
7,ENSG00000108602,ALDH3A1,protein_coding,4439,635,54,70,10533,17,1,22,4,114336,293,0.696827,0,130404
8,ENSG00000152049,KCNE4,protein_coding,33,73,417,128,104,184,1,331,34,15081,243,0.701489,0,16629
9,ENSG00000174028,FAM3C2P,processed_pseudogene,29,224,115,23,69,8168,0,215,95,249,122,0.887762,0,9309


In [49]:
df_tumor3.tail(30)

,gene_id,symbol,gene_type,tumor_1,tumor_2,tumor_3,tumor_4,tumor_5,tumor_6,tumor_7,tumor_8,tumor_9,tumor_10,tumor_11,entropy,entropy_q,total_sum
1552,ENSG00000205542,TMSB4X,protein_coding,10410,56120,50469,23782,14338,247913,25,24118,22609,87685,15153,2.558076,0,552622
1553,ENSG00000169093,ASMTL,protein_coding,474,0,936,0,0,0,55,476,409,1057,608,2.558131,0,4015
1554,ENSG00000169093,ASMTL,protein_coding,474,0,936,819,417,0,55,476,0,1057,0,2.558485,0,4234
1555,ENSG00000169093,ASMTL,protein_coding,474,0,936,0,417,2019,0,0,409,1057,608,2.558664,0,5920
1556,ENSG00000133134,BEX2,protein_coding,218,120,511,948,36,2379,0,428,6,680,538,2.558702,0,5864
1557,ENSG00000169093,ASMTL,protein_coding,474,766,936,0,0,0,55,476,409,1057,0,2.558892,0,4173
1558,ENSG00000169093,ASMTL,protein_coding,0,0,936,0,417,2019,0,476,409,1057,608,2.559029,0,5922
1559,ENSG00000144120,TMEM177,protein_coding,356,214,316,271,153,2433,0,398,145,535,175,2.559243,0,4996
1560,ENSG00000096088,PGC,protein_coding,632,757,1394,10,621,2151,4,20,182,2587,149,2.559266,0,8507
1561,ENSG00000272288,AL451165.2,lncRNA,150,137,196,208,141,335,131,136,126,341,347,2.559293,0,2248


In [50]:
geneid_list = np.unique(df_tumor3['gene_id'])

len(geneid_list)

1018

### GTEx portal

In [68]:
print(len(geneid_list))
np.array(geneid_list[:50])

1018


array(['ENSG00000001626', 'ENSG00000003989', 'ENSG00000004468',
       'ENSG00000005020', 'ENSG00000005059', 'ENSG00000006007',
       'ENSG00000006016', 'ENSG00000006210', 'ENSG00000006534',
       'ENSG00000006747', 'ENSG00000007402', 'ENSG00000008226',
       'ENSG00000008735', 'ENSG00000008853', 'ENSG00000012171',
       'ENSG00000017483', 'ENSG00000018236', 'ENSG00000019102',
       'ENSG00000019186', 'ENSG00000021300', 'ENSG00000021826',
       'ENSG00000022267', 'ENSG00000022556', 'ENSG00000023171',
       'ENSG00000023839', 'ENSG00000025423', 'ENSG00000025708',
       'ENSG00000026103', 'ENSG00000026508', 'ENSG00000029153',
       'ENSG00000034053', 'ENSG00000038002', 'ENSG00000040531',
       'ENSG00000043039', 'ENSG00000047457', 'ENSG00000047662',
       'ENSG00000047936', 'ENSG00000048052', 'ENSG00000049247',
       'ENSG00000051128', 'ENSG00000053918', 'ENSG00000057294',
       'ENSG00000060718', 'ENSG00000060982', 'ENSG00000062038',
       'ENSG00000064199', 'ENSG000000642

In [69]:
GTEX_API = "https://gtexportal.org/api/v2"

In [70]:
def resolve_gtex_gencode_id(gene_id, datasetId="gtex_v8", timeout=60):
    gene_base = str(gene_id).split(".")[0]

    url = f"{GTEX_API}/reference/gene"

    params = {
        "geneId": gene_base,
        "datasetId": datasetId,
    }

    r = requests.get(url, params=params, timeout=timeout)
    r.raise_for_status()

    data = r.json().get("data", [])

    if not data:
        return None

    return data[0].get("gencodeId")

In [74]:


def get_gtex_expression_for_geneid_list(
    gtex_id:str,
    geneid_list:list,
    datasetId:str="gtex_v8",
    page_size:int=1000,
    sleep: float = 0.1,
    timeout:int=60,
    verbose:bool=False,
):
    """
    Download GTEx normal tissue expression for selected genes.
    Returns long-format dataframe:
    geneSymbol | gencodeId | tissueSiteDetailId | sampleId | tpm | log2Tpm
    """

    all_rows = []

    print(">>>", len(geneid_list))
    for i, geneid in enumerate(geneid_list):
        page = 0

        while True:

            gencode_id = resolve_gtex_gencode_id(geneid)
            print(">>>", gencode_id, end=" ")

            if gencode_id is None:
                print("?")
                if verbose: print(f"Nothing found for {geneid}")
                break
            
            url = f"{GTEX_API}/expression/geneExpression"

            params = {
                "datasetId": datasetId,
                "gencodeId": gencode_id,
                "tissueSiteDetailId": gtex_id,
                "page": page,
                "itemsPerPage": page_size,
            }

            r = requests.get(url, params=params, timeout=timeout)
            r.raise_for_status()

            js = r.json()
            data = js.get("data", [])

            if not data:
                print("x")
                if verbose: print(f"Error: could not retrieve data for {gtex_id} for geneid '{geneid}'")
                break

            print("Ok")

            all_rows.extend(data)

            paging = js.get("paging_info", {})
            n_pages = paging.get("numberOfPages", 1)

            page += 1
            if page >= n_pages:
                break

            time.sleep(sleep)

    print("")
    df = pd.DataFrame(all_rows)
    return df

In [75]:
gtex_id

'Lung'

In [76]:
df_gtex = get_gtex_expression_for_geneid_list(
                gtex_id=gtex_id,
                geneid_list=geneid_list,
                datasetId="gtex_v8",
                page_size=1000,
                sleep = 0.1,
                timeout=60)

print(df_gtex.shape)

>>> 1018
>>> ENSG00000001626.14 Ok
>>> ENSG00000003989.17 Ok
>>> ENSG00000004468.12 Ok
>>> ENSG00000005020.12 Ok
>>> ENSG00000005059.15 Ok
>>> ENSG00000006007.11 Ok
>>> ENSG00000006016.10 Ok
>>> ENSG00000006210.6 Ok
>>> ENSG00000006534.15 Ok
>>> ENSG00000006747.14 Ok
>>> ENSG00000007402.11 Ok
>>> ENSG00000008226.19 Ok
>>> ENSG00000008735.13 Ok
>>> ENSG00000008853.16 Ok
>>> ENSG00000012171.19 Ok
>>> ENSG00000017483.14 Ok
>>> ENSG00000018236.14 Ok
>>> ENSG00000019102.11 Ok
>>> ENSG00000019186.9 Ok
>>> ENSG00000021300.13 Ok
>>> ENSG00000021826.14 Ok
>>> ENSG00000022267.16 Ok
>>> ENSG00000022556.15 Ok
>>> ENSG00000023171.16 Ok
>>> ENSG00000023839.10 Ok
>>> ENSG00000025423.11 Ok
>>> ENSG00000025708.13 Ok
>>> ENSG00000026103.21 Ok
>>> ENSG00000026508.17 Ok
>>> ENSG00000029153.14 Ok
>>> ENSG00000034053.14 Ok
>>> ENSG00000038002.8 Ok
>>> ENSG00000040531.14 Ok
>>> ENSG00000043039.6 Ok
>>> ENSG00000047457.13 Ok
>>> ENSG00000047662.4 Ok
>>> ENSG00000047936.10 Ok
>>> ENSG00000048052.21 Ok
>>> ENSG

In [82]:
gdc.root_gtex = create_dir(gdc.ROOT_DATA0, 'GTEx')

In [83]:
fname = "gtex_TPM_%s.tsv"%(psi_id)
pdwritecsv(df_gtex, fname, gdc.root_gtex)

print(len(df_gtex))
df_gtex.head()

991


,data,tissueSiteDetailId,ontologyId,datasetId,gencodeId,geneSymbol,unit,subsetGroup
0,"[4.23, 0.8048, 1.058, 7.783, 3.34, 1.121, 0.37...",Lung,UBERON:0008952,gtex_v8,ENSG00000001626.14,CFTR,TPM,None
1,"[14.6, 9.227, 21.5, 20.61, 5.981, 34.44, 16.79...",Lung,UBERON:0008952,gtex_v8,ENSG00000003989.17,SLC7A2,TPM,None
2,"[9.614, 5.767, 6.255, 4.502, 4.926, 14.01, 10....",Lung,UBERON:0008952,gtex_v8,ENSG00000004468.12,CD38,TPM,None
3,"[26.78, 44.86, 36.63, 42.64, 30.5, 40.02, 34.4...",Lung,UBERON:0008952,gtex_v8,ENSG00000005020.12,SKAP2,TPM,None
4,"[15.37, 10.43, 8.518, 10.34, 9.851, 17.75, 16....",Lung,UBERON:0008952,gtex_v8,ENSG00000005059.15,MCUB,TPM,None


### GTEx download

https://www.gtexportal.org/home/downloads/adult-gtex/overview

In [84]:
def download_file(url, filename_out):
    with requests.get(url, stream=True) as r:
        r.raise_for_status()

        try:
            with open(filename_out, "wb") as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)
        except Exception as e:
            print(f"Error: as writingfile {filename_out}: {e}")



In [101]:
url_counts = (
    "https://storage.googleapis.com/adult-gtex/bulk-gex/v11/rna-seq/"
    "GTEx_Analysis_2025-08-22_v11_RNASeQCv2.4.3_gene_reads.gct.gz"
)

url_tpm = (
    "https://storage.googleapis.com/adult-gtex/bulk-gex/v11/rna-seq/"
    "GTEx_Analysis_2025-08-22_v11_RNASeQCv2.4.3_gene_tpm.gct.gz"
)

url_meta = (
    "https://storage.googleapis.com/adult-gtex/metadata/"
    "GTEx_Analysis_v11_Annotations_SampleAttributesDS.txt"
)
fname_counts = "GTEx_Analysis_2025-08-22_v11_RNASeQCv2.4.3_gene_reads.gct.gz"
fname_meta   = "GTEx_Analysis_v11_Annotations_SampleAttributesDS.tsv"

filename_counts = gdc.root_gtex / fname_counts
filename_meta   = gdc.root_gtex / fname_meta

# --> download manually at https://www.gtexportal.org/home/downloads/adult-gtex/bulk_tissue_expression
#                      and https://www.gtexportal.org/home/downloads/adult-gtex/metadata

# download_file(url_counts, filename_counts)
# download_file(url_meta, filename_meta)

### Reading GTEx count super-file

In [94]:
# load matrix'1
dfcounts = pdreadcsv(fname_counts, gdc.root_gtex, skiprows=2)
print(len(dfcounts))
dfcounts.head(3)

74628


,Name,Description,GTEX-1117F-0005-SM-HL9SH,GTEX-1117F-0011-R10b-SM-GI4VE,GTEX-1117F-0011-R11b-SM-GIN8R,GTEX-1117F-0011-R2b-SM-GI4VL,GTEX-1117F-0011-R3a-SM-GJ3PJ,GTEX-1117F-0011-R4b-SM-GI4VM,GTEX-1117F-0011-R5a-SM-GI4VW,GTEX-1117F-0011-R6a-SM-GI4VX,...,GTEX-ZZPU-1326-SM-5GZWS,GTEX-ZZPU-1426-SM-5GZZ6,GTEX-ZZPU-1826-SM-5E43L,GTEX-ZZPU-2126-SM-5EGIU,GTEX-ZZPU-2226-SM-5EGIV,GTEX-ZZPU-2326-SM-GOQYU,GTEX-ZZPU-2426-SM-5E44I,GTEX-ZZPU-2526-SM-GOQZ3,GTEX-ZZPU-2626-SM-5E45Y,GTEX-ZZPU-2726-SM-5NQ8O
0,ENSG00000290825.2,DDX11L16,0,0,1,0,0,3,0,1,...,1,0,0,0,0,0,0,1,0,0
1,ENSG00000223972.6,DDX11L1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,ENSG00000310526.1,WASH7P,90,137,452,69,95,174,269,261,...,397,231,215,266,152,474,139,896,88,213


In [103]:
# load metadata
dfmeta = pdreadcsv(fname_meta, gdc.root_gtex)
print(len(dfmeta))
dfmeta.head(3)

48231


,SAMPID,SMATSSCR,SMCENTER,SMPTHNTS,SMRIN,SMTS,SMTSD,SMUBRID,SMTSISCH,SMTSPAX,...,SMSHRTRT,SMSMRDHQ,SMSMRTHQ,SMPRERDHQ,SMPRERTHQ,SMSMGNDT,SMPREGNDT,SMRDLNMN,SMRDLNMD,SMRDLNSD
0,BMS-X4LF-0126-SM-4JBHL,NaN,B1,NaN,7.5,Thyroid,Thyroid,UBERON:0002046,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,BMS-X4LF-0226-SM-4JBJ3,NaN,B1,NaN,6.9,Blood Vessel,Artery - Pulmonary,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,BMS-X4LF-0326-SM-4JBIR,NaN,B1,NaN,7.4,Muscle,Muscle - Skeletal,UBERON:0011907,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [122]:
# select lung samples
lung_samples = dfmeta.loc[dfmeta["SMTSD"] == "Lung", "SAMPID"].to_list()
print(len(lung_samples))
lung_samples[:5]

1892


['GTEX-1117F-1026-SM-GW2CO',
 'GTEX-1117F-1026-SM-LLLJU',
 'GTEX-111CU-0326-SM-5GZXO',
 'GTEX-111CU-0326-SM-F1XB2',
 'GTEX-111FC-1126-SM-5GZWU']

In [117]:
dfcounts.columns

Index(['Name', 'Description', 'GTEX-1117F-0005-SM-HL9SH',
       'GTEX-1117F-0011-R10b-SM-GI4VE', 'GTEX-1117F-0011-R11b-SM-GIN8R',
       'GTEX-1117F-0011-R2b-SM-GI4VL', 'GTEX-1117F-0011-R3a-SM-GJ3PJ',
       'GTEX-1117F-0011-R4b-SM-GI4VM', 'GTEX-1117F-0011-R5a-SM-GI4VW',
       'GTEX-1117F-0011-R6a-SM-GI4VX',
       ...
       'GTEX-ZZPU-1326-SM-5GZWS', 'GTEX-ZZPU-1426-SM-5GZZ6',
       'GTEX-ZZPU-1826-SM-5E43L', 'GTEX-ZZPU-2126-SM-5EGIU',
       'GTEX-ZZPU-2226-SM-5EGIV', 'GTEX-ZZPU-2326-SM-GOQYU',
       'GTEX-ZZPU-2426-SM-5E44I', 'GTEX-ZZPU-2526-SM-GOQZ3',
       'GTEX-ZZPU-2626-SM-5E45Y', 'GTEX-ZZPU-2726-SM-5NQ8O'],
      dtype='object', length=19790)

In [118]:
cols = ["Name", "Description"] + list(lung_samples)

In [121]:
good_cols = [x for x in cols if x in dfcounts.columns]

In [ ]:
df2 = dfcounts.loc[:, good_cols].copy()


KeyError: "['GTEX-1117F-1026-SM-GW2CO', 'GTEX-1117F-1026-SM-LLLJU', 'GTEX-111CU-0326-SM-F1XB2', 'GTEX-111FC-1126-SM-LUBBH', 'GTEX-111VG-0726-SM-GADV8', 'GTEX-111YS-0626-SM-F1CL6', 'GTEX-1122O-0126-SM-F1BZV', 'GTEX-1128S-0726-SM-EUKOO', 'GTEX-113JC-1326-SM-GW1F2', 'GTEX-113JC-1326-SM-LUBBT', 'GTEX-117YW-0526-SM-EUJFZ', 'GTEX-117YX-1326-SM-F3JKO', 'GTEX-1192W-0626-SM-GW224', 'GTEX-1192X-1326-SM-GW191', 'GTEX-11DXZ-0726-SM-GW23R', 'GTEX-11DYG-0626-SM-GW1XU', 'GTEX-11DZ1-0426-SM-EUPBM', 'GTEX-11EI6-0826-SM-CYKV8', 'GTEX-11EMC-0126-SM-F1KDQ', 'GTEX-11EQ9-0226-SM-F1FCM', 'GTEX-11GSP-0726-SM-9VXZ4', 'GTEX-11GSP-0726-SM-9WPNU', 'GTEX-11GSP-0726-SM-GW2CL', 'GTEX-11I78-0126-SM-DECQN', 'GTEX-11I78-0126-SM-GW226', 'GTEX-11ILO-0726-SM-5HL5I', 'GTEX-11ILO-0726-SM-GM8M3', 'GTEX-11NSD-0326-SM-EV6WG', 'GTEX-11NUK-0826-SM-EUKLA', 'GTEX-11NUK-0826-SM-LVTBX', 'GTEX-11NV4-1126-SM-F5WG8', 'GTEX-11NV4-1126-SM-GW17Z', 'GTEX-11NV4-1126-SM-LUBC6', 'GTEX-11O72-1326-SM-GEGUH', 'GTEX-11OF3-1126-SM-F1KN5', 'GTEX-11

In [ ]:
df2.rename(columns={"Name": "ensemblid", "Description": "symbol"}, inplace=True)

print(len(df2))
df2.head(3)